# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [10]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /Users/amanz/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/amanz/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [19]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [20]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [21]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Use-Case Data!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [22]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [24]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [25]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [26]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 64, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [27]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node '885e30'. Skipping!
Property 'summary' already exists in node '983401'. Skipping!
Property 'summary' already exists in node '89a86e'. Skipping!
Property 'summary' already exists in node '675160'. Skipping!
Property 'summary' already exists in node '607ed4'. Skipping!
Property 'summary' already exists in node 'a7bb65'. Skipping!
Property 'summary' already exists in node '15d9a7'. Skipping!
Property 'summary' already exists in node '83d0a4'. Skipping!
Property 'summary' already exists in node '6579d2'. Skipping!
Property 'summary' already exists in node '1b6322'. Skipping!
Property 'summary' already exists in node '80a694'. Skipping!
Property 'summary' already exists in node '7bc4f7'. Skipping!
Property 'summary' already exists in node '35dca4'. Skipping!
Property 'summary' already exists in node '536f00'. Skipping!
Property 'summary' already exists in node '9867a9'. Skipping!
Property 'summary' already exists in node 'a6ff4b'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/48 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '675160'. Skipping!
Property 'summary_embedding' already exists in node '983401'. Skipping!
Property 'summary_embedding' already exists in node '885e30'. Skipping!
Property 'summary_embedding' already exists in node '6579d2'. Skipping!
Property 'summary_embedding' already exists in node '607ed4'. Skipping!
Property 'summary_embedding' already exists in node '89a86e'. Skipping!
Property 'summary_embedding' already exists in node '35dca4'. Skipping!
Property 'summary_embedding' already exists in node '1b6322'. Skipping!
Property 'summary_embedding' already exists in node '15d9a7'. Skipping!
Property 'summary_embedding' already exists in node '83d0a4'. Skipping!
Property 'summary_embedding' already exists in node 'a7bb65'. Skipping!
Property 'summary_embedding' already exists in node '7bc4f7'. Skipping!
Property 'summary_embedding' already exists in node '536f00'. Skipping!
Property 'summary_embedding' already exists in node '9867a9'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 86, relationships: 713)

We can save and load our knowledge graphs as follows.

In [28]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 86, relationships: 713)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [29]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [30]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.


n ragas, the query synthesizers are LLM-based components that generate test queries for your dataset so you can evaluate RAG systems. Each synthesizer creates queries of a different “style” and difficulty:

1. SingleHopSpecificQuerySynthesizer

Purpose: Generates straightforward factoid questions that can be answered by retrieving a single chunk of context.

Example: If the context is “Grace Church is located in Houston, TX.” → it might generate “Where is Grace Church located?”

Use case: Tests whether your retriever+LLM can handle direct, single-document lookups.

2. MultiHopAbstractQuerySynthesizer

Purpose: Generates higher-level, reasoning-oriented queries that require combining multiple pieces of information from across the dataset, but phrased in a more open-ended/abstract way.

Example: If the dataset contains “Grace Church is located in Houston” and “Houston is the largest city in Texas” → it might ask “Which major Texas city is home to Grace Church?”

Use case: Tests whether the system can connect facts and reason across documents, not just pull verbatim text.

3. MultiHopSpecificQuerySynthesizer

Purpose: Also generates multi-document, reasoning queries, but these are concrete and fact-specific (not broad or abstract).

Example: If the dataset has “Pastor John leads Grace Church” and “Pastor John previously led a church in Dallas” → it might generate “Which city did Grace Church’s pastor work in before Houston?”

Use case: Tests whether the system can stitch together specific factual details that span multiple contexts.


Finally, we can use our `TestSetGenerator` to generate our testset!

In [31]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,How is the adoption and usage of ChatGPT proje...,[Introduction ChatGPT launched in November 202...,"By July 2025, 18 billion messages are being se...",single_hop_specifc_query_synthesizer
1,What is the US?,[Table 1: ChatGPT daily message counts (millio...,The context does not provide a specific defini...,single_hop_specifc_query_synthesizer
2,What SOC means in work stuff?,[Variation by Occupation Figure 23 presents va...,Variation by Occupation Figure 23 presents var...,single_hop_specifc_query_synthesizer
3,What does Personal Reflection mean in this con...,[Conclusion This paper studies the rapid growt...,"In this context, Personal Reflection refers to...",single_hop_specifc_query_synthesizer
4,How do the changes in ChatGPT message categori...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,The data shows that total ChatGPT messages inc...,multi_hop_abstract_query_synthesizer
5,How do the changes in ChatGPT message activity...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,"Between June 2024 and June 2025, ChatGPT usage...",multi_hop_abstract_query_synthesizer
6,How do user demographics and message classific...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"The context indicates that as of July 2025, ap...",multi_hop_abstract_query_synthesizer
7,How does OpenAI's development and widespread a...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,The context indicates that since its launch in...,multi_hop_specific_query_synthesizer
8,How does the rapid growth of ChatGPT in the US...,[<1-hop>\n\nConclusion This paper studies the ...,"The context indicates that by July 2025, ChatG...",multi_hop_specific_query_synthesizer
9,How do the details in Appendix D support the p...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The information in Appendix D provides a detai...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [32]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/39 [00:00<?, ?it/s]

Property 'summary' already exists in node 'd04bfd'. Skipping!
Property 'summary' already exists in node '66bc31'. Skipping!
Property 'summary' already exists in node '9ce55e'. Skipping!
Property 'summary' already exists in node '7b7503'. Skipping!
Property 'summary' already exists in node 'a40a87'. Skipping!
Property 'summary' already exists in node '50fda6'. Skipping!
Property 'summary' already exists in node '9ee8d5'. Skipping!
Property 'summary' already exists in node '5c0b77'. Skipping!
Property 'summary' already exists in node '7495c9'. Skipping!
Property 'summary' already exists in node '2b3276'. Skipping!
Property 'summary' already exists in node '2f1cd8'. Skipping!
Property 'summary' already exists in node 'b06ddc'. Skipping!
Property 'summary' already exists in node '5fa9ea'. Skipping!
Property 'summary' already exists in node '5b8846'. Skipping!
Property 'summary' already exists in node '8adfcf'. Skipping!
Property 'summary' already exists in node 'e680b5'. Skipping!
Property

...

Property 'summary_embedding' already exists in node '50fda6'. Skipping!
Property 'summary_embedding' already exists in node 'a40a87'. Skipping!
Property 'summary_embedding' already exists in node '66bc31'. Skipping!
Property 'summary_embedding' already exists in node '9ce55e'. Skipping!
Property 'summary_embedding' already exists in node '7b7503'. Skipping!
Property 'summary_embedding' already exists in node 'd04bfd'. Skipping!
Property 'summary_embedding' already exists in node '2f1cd8'. Skipping!
Property 'summary_embedding' already exists in node '2b3276'. Skipping!
Property 'summary_embedding' already exists in node '9ee8d5'. Skipping!
Property 'summary_embedding' already exists in node '7495c9'. Skipping!
Property 'summary_embedding' already exists in node 'e680b5'. Skipping!
Property 'summary_embedding' already exists in node 'b06ddc'. Skipping!
Property 'summary_embedding' already exists in node '8adfcf'. Skipping!
Property 'summary_embedding' already exists in node '5c0b77'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

In [33]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What significance does November 2022 hold in r...,[Introduction ChatGPT launched in November 202...,Introduction ChatGPT launched in November 2022.,single_hop_specifc_query_synthesizer
1,"What does Eloundou et al., 2025 say about AI a...",[Introduction ChatGPT launched in November 202...,"Eloundou et al., 2025 discusses the effects of...",single_hop_specifc_query_synthesizer
2,What is the significance of June 2025 in the c...,[Table 1: ChatGPT daily message counts (millio...,"The report provides data ending on June 26th, ...",single_hop_specifc_query_synthesizer
3,What is the significance of June 2025 in the c...,[Table 1: ChatGPT daily message counts (millio...,"The report provides data ending on June 26th, ...",single_hop_specifc_query_synthesizer
4,What SOC means in chatgpt usage data?,[Variation by Occupation Figure 23 presents va...,Variation by Occupation Figure 23 presents var...,single_hop_specifc_query_synthesizer
5,Could you please explain the significance of t...,[Variation by Occupation Figure 23 presents va...,"In the provided context, the term 'Section' re...",single_hop_specifc_query_synthesizer
6,Whay data privacy limts and ocpation varation ...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The context indicates that due to privacy-pres...,multi_hop_abstract_query_synthesizer
7,How do variations in ChatGPT usage by occupati...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The data indicates that users in highly paid p...,multi_hop_abstract_query_synthesizer
8,How does message classifcation and taxonomy he...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,Message classification and taxonomy are used t...,multi_hop_abstract_query_synthesizer
9,how ChatGPT message volume and usage patterns ...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,The context shows that ChatGPT daily message c...,multi_hop_abstract_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [34]:
from langsmith import Client

client = Client()

dataset_name = "Use Case Synthetic Data - AIE8"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

LangSmithConflictError: Conflict for /datasets. HTTPError('409 Client Error: Conflict for url: https://api.smith.langchain.com/datasets', '{"detail":"Dataset with this name already exists."}')

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [35]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

NameError: name 'langsmith_dataset' is not defined

## Basic RAG Chain

Time for some RAG!


In [36]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [37]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [38]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [39]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG"
)

In [40]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [41]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [42]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [43]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [44]:
rag_chain.invoke({"question" : "What are people doing with AI these days?"})

"Based on the provided context, people are using AI in a variety of ways both at work and outside of work. Specifically, generative AI like ChatGPT is highly flexible and is used to perform workplace tasks that either augment or automate human labor. Users seek different types of outputs from AI, including writing, software code, spreadsheets, and other digital products. AI serves roles such as co-workers producing output or co-pilots giving advice and improving human problem-solving productivity. Additionally, while some users request information and advice similar to traditional web searches, generative AI's ability to produce creative and functional digital work distinguishes it from existing technologies. \n\nHence, people are using AI to:\n\n- Augment or automate work tasks\n- Generate writing and code\n- Create spreadsheets and other digital products\n- Improve productivity through advice and problem-solving assistance\n- Seek information and advice in a conversational manner\n\n

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [45]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [46]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dopeness_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this response dope, lit, cool, or is it just a generic response?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- qa_evaluator → correctness
- labeled_helpfulness_evaluator → usefulness/helpfulness
- dopeness_evaluator → stylistic “coolness”

## LangSmith Evaluation

In [47]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'crushing-collar-5' at:
https://smith.langchain.com/o/0c815d0a-6da4-4066-a03c-dad8a19efc56/datasets/16705b16-d06a-4c0d-8b6c-635e3abb15e4/compare?selectedSessions=4ad6a83d-7873-4f12-aa9c-1f446621005e




0it [00:00, ?it/s]

KeyboardInterrupt: 

## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [48]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [49]:
rag_documents = docs

In [50]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

In [36]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

Your retriever only sees chunks, not the entire document. So the way you cut up your text directly controls what context gets embedded, retrieved, and passed to the LLM.

1. Too small (tiny chunks)

Pros:

Fine-grained retrieval (retriever grabs exactly the relevant sentence/paragraph).

Less wasted tokens (no huge irrelevant context).

Cons:

Important context may get split across chunks → retriever misses part of the answer.

LLM might lack enough info if only one small chunk is returned.

More embeddings to store/query → higher cost + slower retrieval.

2. Too large (big chunks)

Pros:

Fewer splits → less chance of separating context that belongs together.

Retrieval more likely to contain the full answer in a single chunk.

Cons:

Embeddings become “blurry”: large chunks embed lots of unrelated sentences → similarity search gets noisier.

More irrelevant text per retrieved chunk (hurts answer accuracy).

Higher token cost when sending to LLM.

3. The sweet spot

Good chunking balances:

Semantic completeness (answerable from one or a few chunks).

Search precision (embeddings represent a coherent idea, not a whole page).

Efficiency (not flooding the LLM with irrelevant context).

That’s why people often do:

“semantic chunking” (split by headings/paragraphs instead of fixed tokens).

“parent–child” retrieval (store smaller child chunks for retrieval, but return the larger parent section).

overlapping windows (so context isn’t lost at chunk boundaries).

In [37]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [38]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [39]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [40]:
dopeness_rag_chain.invoke({"question" : "How are people using AI to make money?"})

"Alright, buckle up for this AI-powered cash flow breakdown straight from the digital frontlines! People aren't just bossing around AI to get repetitive tasks done—they’re tagging ChatGPT as their slick advisor and research sidekick, leveling up decision-making like seasoned pros. Especially in knowledge-heavy gigs, AI’s boosting productivity by supercharging the quality of choices workers make, not just cranking out tasks on autopilot.\n\nThink of AI as the ultimate co-pilot: it’s dishing out practical guidance, info-seeking wizardry, and writing mojo—about 80% of all ChatGPT magic falls here. Businesses and freelancers alike are tapping that to sharpen their moves, craft killer content, and strategize smarter—not just automate grunt work, but actually *augment* human hustle.\n\nAnd here’s the kicker: US users value this AI drop so much that they’d demand a cool $98 each to skip using generative AI for just a month—translating into a mind-blowing consumer surplus north of $97 billion 

Finally, we can evaluate the new chain on the same test set!

In [41]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'left-thunder-62' at:
https://smith.langchain.com/o/340cd80b-3296-5752-9a9e-58582118073a/datasets/205b64dd-313a-43ea-9351-176895cb52ed/compare?selectedSessions=6732ad11-fe5f-45c3-a467-1815a9237214




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,US ChatGPT use more in work or non-work in US ...,"Alright, here’s the ultra-dope lowdown on how ...",None,"According to the context, in the US, about 70%...",0,0,1,5.062981,700061d8-6a9b-4c1d-8a2e-7840a86f0d71,324720e8-b8ab-425e-913f-871adad3c5ef
1,Hw US ChatGPT usage in US is mostly non-work r...,"Yo, let’s break this down with some serious sa...",None,"The context indicates that in the US, as of Ju...",1,1,1,4.983939,fccf777c-6dbc-439d-9cae-f5450c8b8a59,984cab1a-363e-43e4-8047-74af714bf877
2,"How do the 700 million users of ChatGPT, as of...","Yo, let’s vibe with the facts here—700 million...",None,"By July 2025, ChatGPT had been used weekly by ...",1,1,1,5.771400,73cb07e7-c146-481e-9975-3dc997139da4,d4a1bcba-398f-4124-9ee1-52bf82d778aa
3,how do 700 million users use chatgpt for data ...,"Alright, let's unpack this with some next-leve...",None,The provided context does not explicitly detai...,1,1,1,3.929451,2697091e-e8df-4c48-a06d-f55f1c3696b5,df4c2137-4727-4165-bec3-cadd4b84814f
4,How does the increase in non-work mesages (M) ...,"Yo, let's break this down with pure AI swagger...",None,The data shows that non-work messages increase...,1,1,1,5.382964,7254bf50-5e64-431c-b2d4-178165250c64,62a8bd74-d98d-40ca-8714-f7cd090233d1
5,How does the increase in non-work message data...,"Yo, here’s the straight-up rad breakdown: Betw...",None,The data shows that non-work messages increase...,1,1,1,5.821866,bc50905e-2d42-4193-b298-854396042884,bebe9ede-0338-4703-beef-8b4128163275
6,How does the message volume comparison between...,"Yo, buckle up for this data deep-dive with som...",None,"The context shows that in June 2024, total mes...",1,1,1,4.310435,e8c59127-5778-46db-aa0d-f46a136c95c2,821cd3da-a594-4fb1-b017-9167d5aecf3a
7,How do the large language models (LLMs) like C...,"Alright, let’s crank this up to eleven and div...",None,"ChatGPT, launched in November 2022 and based o...",1,0,1,11.313386,5da42559-1aa0-4720-9648-d4c8abcaf4b6,d21c3e5b-d886-4c75-9db3-3689fc635eab
8,How does Writing relate to the overall use of ...,"Yo, buckle up because the way Writing flexes i...",None,"Writing is by far the most common work use, ac...",1,1,1,4.778410,fa3854f5-cab3-4f5a-b48c-63a85836d017,d5bd449c-1c06-41da-a4c9-f756f06672da
9,Other Professional what does it mean in ChatGP...,"Alright, buckle up, because diving into the re...",None,Variation by Occupation Figure 23 reports resu...,1,1,1,7.618504,40c1ed9e-a7b2-4ef1-b2f4-dcf079204be8,f3808d96-2a0d-484f-be1e-ae340864be08


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

🔹 What stayed the same

Dataset: Both runs used the same ragas-generated synthetic test set (“Use Case Synthetic Data – AIE8”).

Evaluation metrics: Correctness, Openness, Helpfulness, Latency, Tokens, plus your custom “Dopeness”.

LLM family: Both runs used GPT-4.1-mini for answering, GPT-4.1 for evals.

So differences come from prompting + chunking + embeddings.

🔹 Key differences in setup
Left run (“best-butter-39”)

Chunk size: 500 tokens (with overlap 50).

Embedding model: text-embedding-3-small.

Prompt: Strict RAG style: “If you cannot answer based on context, say ‘I don’t know’.”

Retriever k: 10.

Right run (“abandoned-value-22”)

Chunk size: 1000 tokens (with overlap 50).

Embedding model: upgraded to text-embedding-3-large.

Prompt: Adds stylistic constraint: “Make your answer rad, ensure high levels of dopeness. Do not be generic.”

Retriever k: default (not explicitly set, so likely k=4).

🔹 Explaining the metric shifts
1. Correctness

Left: 0.8333 avg

Right: 0.9167 avg → improved

Why:

Larger embeddings (3-large) captured semantic relationships better than 3-small, so retrieval was stronger.

Larger chunks (1000 vs 500) meant more context per retrieved doc, reducing “I don’t know” responses.

Even though stylistic prompting was added, correctness remained high because the answers still contained the factual content.

2. Helpfulness

Left: 0.75 avg

Right: 0.6667 avg → dropped

Why:

The “rad / dope” prompt forced creative phrasing. While still factually correct, some responses were less concise and less user-oriented, so the evaluator judged them less “helpful.”

Example: “Boom! Let’s dive into the raw juice…” vs. a plain informative answer — correct but less directly helpful.

3. Dopeness

Left: 0.00 avg

Right: 1.00 avg → huge jump

Why:

Prompt engineering explicitly pushed for “dope, lit, cool” responses. The evaluator criteria aligned perfectly with this new style, so scores maxed out.

This shows how evaluator framing + prompting can dramatically swing subjective metrics.

4. Latency

Left: ~5.45s median (p50)

Right: ~7.28s median (p50) → slower

Why:

Larger embedding model (3-large) = slower retrieval step.

Bigger chunks (1000 tokens) = more tokens passed into the LLM.

Stylized responses = longer generations.

5. Tokens

Left: 44,191 total

Right: 26,273 total → dropped

Why:

Even though chunks were bigger, the retriever defaulted to fewer results (k=4 vs. k=10).

So less total context was being retrieved and sent into the LLM each time.

This cut total token usage nearly in half.

🔹 Big picture takeaways

Better embeddings (3-large) → correctness improved.

Fewer retrieved chunks (k=4 vs. 10) → tokens dropped dramatically.

Stylized prompting → “dopeness” maxed out but helpfulness fell.

Larger chunks (1000) → helped correctness but slowed latency.


![alt text](1.png "Title")